In [ ]:
!pip install qiskit-aer

In [ ]:
import math
import time
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.circuit.library import MCXGate

# ---------------------------------------------------------------------------
# config
# ---------------------------------------------------------------------------
MAX_QUBITS = 29
SAFE_SIM_QUBITS = 24
BYTES_PER_AMPLITUDE = 16
RAM_BUDGET_BYTES = 8 * 1024 ** 3

QUERY_SIZES = [10, 1_000, 100_000, 500_000, 1_000_000, 2_000_000]

# ---------------------------------------------------------------------------

def qubits_needed(n_items: int) -> int:
    return max(1, math.ceil(math.log2(n_items)))


def statevector_gb(n_qubits: int) -> float:
    return (2 ** n_qubits * BYTES_PER_AMPLITUDE) / 1024 ** 3


def fits_in_ram(n_qubits: int) -> bool:
    return 2 ** n_qubits * BYTES_PER_AMPLITUDE <= RAM_BUDGET_BYTES


# ---------------------------------------------------------------------------
# 1. classical linear search
# ---------------------------------------------------------------------------
def classical_linear_search(items, target) -> dict:
    comparisons = 0
    found_index = None
    for i, val in enumerate(items):
        comparisons += 1
        if val == target:
            found_index = i
            break
    return {
        "found_index": found_index,
        "queries_used": comparisons,
        "worst_case_queries": len(items),
        "avg_case_queries": (len(items) + 1) / 2,
    }


# ---------------------------------------------------------------------------
# 2. grover's quantum search
# ---------------------------------------------------------------------------
def build_grover_circuit(n_qubits: int, marked_state: str, max_iterations: int = 2000):
    n = n_qubits
    N = 2 ** n
    true_iterations = max(1, round((math.pi / 4) * math.sqrt(N)))
    sim_iterations = min(true_iterations, max_iterations)

    if n_qubits == 1:
        qc_step = QuantumCircuit(1, name="GroverStep")
        if marked_state == '0':
            qc_step.x(0)
            qc_step.z(0)
            qc_step.x(0)
        elif marked_state == '1':
            qc_step.z(0)
        else:
            raise ValueError("Invalid marked_state for 1 qubit. Must be '0' or '1'.")
        qc_step.z(0)
        qc_step.h(0)
        circuit = QuantumCircuit(1, 1)
        circuit.h(0)
        grover_step_gate = qc_step.to_gate(label="GroverStep")
        circuit.append(grover_step_gate.repeat(sim_iterations), [0])
        circuit.measure(0, 0)
        return circuit, true_iterations, sim_iterations

    n_ctrl = n - 1
    mcx_gate = MCXGate(n_ctrl)

    def grover_step_multi_qubit():
        qc = QuantumCircuit(n, name="GroverStep")
        data = list(range(n))
        zero_positions = [i for i, b in enumerate(reversed(marked_state)) if b == "0"]
        if zero_positions:
            qc.x(zero_positions)
        qc.h(n - 1)
        qc.append(mcx_gate, list(range(n_ctrl)) + [n-1])
        qc.h(n - 1)
        if zero_positions:
            qc.x(zero_positions)
        qc.h(data)
        qc.x(data)
        qc.h(n - 1)
        qc.append(mcx_gate, list(range(n_ctrl)) + [n-1])
        qc.h(n - 1)
        qc.x(data)
        qc.h(data)
        return qc

    step = grover_step_multi_qubit()
    circuit = QuantumCircuit(n, n)
    circuit.h(range(n))

    grover_step_gate = step.to_gate(label="GroverStep")
    circuit.append(grover_step_gate.repeat(sim_iterations), range(n))

    circuit.measure(range(n), range(n))
    return circuit, true_iterations, sim_iterations


def run_grover_simulation(n_qubits: int, marked_state: str, shots: int = 2048) -> dict:
    circuit, true_iterations, sim_iterations = build_grover_circuit(n_qubits, marked_state)
    sim = AerSimulator(method="statevector")
    t0 = time.time()
    transpiled_circuit = transpile(circuit, sim)
    result = sim.run(transpiled_circuit, shots=shots).result()
    elapsed = time.time() - t0
    counts = result.get_counts()
    hits = counts.get(marked_state, 0)
    return {
        "true_grover_iterations": true_iterations,
        "simulated_iterations": sim_iterations,
        "shots": shots,
        "measured_success_prob": hits / shots,
        "sim_seconds": elapsed,
        "statevector_gb": statevector_gb(n_qubits),
    }


def theoretical_grover(n_qubits: int) -> dict:
    N = 2 ** n_qubits
    iterations = max(1, round((math.pi / 4) * math.sqrt(N)))
    theta = math.asin(1 / math.sqrt(N))
    return {
        "grover_iterations": iterations,
        "theoretical_success_prob": math.sin((2 * iterations + 1) * theta) ** 2,
        "statevector_gb": statevector_gb(n_qubits),
    }


# ---------------------------------------------------------------------------
# 3. comparison table: N = 10 ... 2^29
# ---------------------------------------------------------------------------
def compare_row(n_items: int) -> dict:
    n_qubits = qubits_needed(n_items)
    classical_worst = n_items
    classical_avg = (n_items + 1) / 2
    quantum_theoretical = max(1, round((math.pi / 4) * math.sqrt(n_items)))
    return {
        "N": n_items,
        "qubits_needed": n_qubits,
        "classical_worst_case_queries": classical_worst,
        "classical_avg_case_queries": classical_avg,
        "quantum_queries_theoretical": quantum_theoretical,
        "speedup_worst_vs_quantum": round(classical_worst / quantum_theoretical, 2),
        "statevector_ram_needed_GB": round(statevector_gb(n_qubits), 6),
        "can_actually_simulate_in_4GB": fits_in_ram(n_qubits) and n_qubits <= SAFE_SIM_QUBITS,
    }


def main():
    header = (f"{'N':>12} | {'qubits':>6} | {'classical(worst)':>17} | {'classical(avg)':>15} "
              f"| {'quantum(theory)':>16} | {'speedup':>8} | {'RAM needed(GB)':>15} | simulated?")
    print(header)
    print("-" * len(header))

    for n_items in QUERY_SIZES:
        row = compare_row(n_items)
        will_sim = row["can_actually_simulate_in_4GB"]
        print(f"{row['N']:>12} | {row['qubits_needed']:>6} | "
              f"{row['classical_worst_case_queries']:>17} | {row['classical_avg_case_queries']:>15.1f} | "
              f"{row['quantum_queries_theoretical']:>16} | {row['speedup_worst_vs_quantum']:>8} | "
              f"{row['statevector_ram_needed_GB']:>15} | "
              f"{'YES (real Aer run)' if will_sim else 'NO (theory only)'}")

        if will_sim:
            marked = "1" * row["qubits_needed"]
            sim = run_grover_simulation(row["qubits_needed"], marked)
            capped_note = (" (capped for demo speed)"
                            if sim["simulated_iterations"] < sim["true_grover_iterations"] else "")
            print(f"    -> Real simulation: ran {sim['simulated_iterations']}/"
                  f"{sim['true_grover_iterations']} Grover iterations{capped_note}, "
                  f"measured success_prob={sim['measured_success_prob']:.4f}, "
                  f"runtime={sim['sim_seconds']:.3f}s, "
                  f"statevector={sim['statevector_gb']:.6f} GB")
        else:
            th = theoretical_grover(row["qubits_needed"])
            print(f"    -> Theory only (needs {th['statevector_gb']:.2f} GB statevector, "
                  f"over the 4GB sandbox limit): {th['grover_iterations']} iterations, "
                  f"theoretical_success_prob={th['theoretical_success_prob']:.4f}")
        print()


if __name__ == "__main__":
    main()